<a href="https://colab.research.google.com/github/pcitrabbb/Aplikasi-Login/blob/main/ecommerce_dashboard_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce Dashboard — Streamlit di Google Colab

## STEP 1 — Install Dependensi

In [13]:
!pip install streamlit plotly --quiet
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print('Semua dependensi berhasil diinstall')
!cloudflared --version

(Reading database ... 118198 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) over (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...
Semua dependensi berhasil diinstall
cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


## STEP 2 — Upload File CSV


In [14]:
from google.colab import files
import os

uploaded = files.upload()

for fname in uploaded:
    size_mb = len(uploaded[fname]) / (1024 * 1024)
    print(f'File berhasil diupload: {fname} ({size_mb:.1f} MB)')

if os.path.exists('main_data.csv'):
    import pandas as pd
    df_check = pd.read_csv('main_data.csv', nrows=2)
    print(f'Kolom terdeteksi: {len(df_check.columns)} kolom')
    print(f' File siap digunakan!')
else:
    print(' File salah.csv')

Saving main_data.csv to main_data (2).csv
File berhasil diupload: main_data (2).csv (57.3 MB)
Kolom terdeteksi: 36 kolom
 File siap digunakan!


## STEP 3 — Buat File dashboard.py

In [15]:
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="E-Commerce Dashboard",
    page_icon="🛒",
    layout="wide",
    initial_sidebar_state="expanded",
)

@st.cache_data
def load_data(path):
    df = pd.read_csv(path)
    date_cols = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["purchase_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
    df["purchase_year"]  = df["order_purchase_timestamp"].dt.year
    df["delivery_days"]  = (
        df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
    ).dt.days
    return df

st.sidebar.image("https://img.icons8.com/color/96/shopping-cart.png", width=64)
st.sidebar.title("Load Data")
uploaded = st.sidebar.file_uploader("Upload main_data.csv", type=["csv"])

if uploaded:
    df = load_data(uploaded)
    st.sidebar.success(f" File loaded: {len(df):,} rows")
else:
    try:
        df = load_data("main_data.csv")
        st.sidebar.info("Loaded: main_data.csv")
    except FileNotFoundError:
        st.warning(" Upload file CSV melalui sidebar kiri.")
        st.stop()

st.sidebar.markdown("---")
st.sidebar.title(" Filter")
years = sorted(df["purchase_year"].dropna().unique().astype(int).tolist())
sel_years = st.sidebar.multiselect("Tahun Pembelian", years, default=years)
statuses = df["order_status"].dropna().unique().tolist()
sel_status = st.sidebar.multiselect("Status Order", statuses, default=["delivered"])

df_f = df[
    df["purchase_year"].isin(sel_years) &
    df["order_status"].isin(sel_status)
].copy()

st.markdown("""
<h1 style=\'text-align:center; color:#2c3e50;\'>
    🛒 E-Commerce Sales Dashboard
</h1>
<p style=\'text-align:center; color:#7f8c8d; font-size:16px;\'>
    Analisis data penjualan online — Olist Brazilian E-Commerce Dataset
</p><hr>""", unsafe_allow_html=True)

total_orders  = df_f["order_id"].nunique()
total_revenue = df_f["payment_value"].sum()
avg_review    = df_f["review_score"].mean()
avg_delivery  = df_f["delivery_days"].mean()
total_prods   = df_f["product_id"].nunique()
total_sellers = df_f["seller_id"].nunique()

c1,c2,c3,c4,c5,c6 = st.columns(6)
c1.metric("Total Order",    f"{total_orders:,.0f}")
c2.metric("Total Revenue",  f"R$ {total_revenue:,.0f}")
c3.metric("Avg Review",     f"{avg_review:.2f}/5")
c4.metric("Avg Pengiriman", f"{avg_delivery:.1f} hari")
c5.metric("Produk Unik",   f"{total_prods:,}")
c6.metric("Seller Aktif",   f"{total_sellers:,}")
st.markdown("<hr>", unsafe_allow_html=True)

cl, cr = st.columns([2, 1])
with cl:
    st.subheader(" Tren Pendapatan Bulanan")
    rev = df_f.groupby("purchase_month")["payment_value"].sum().reset_index().sort_values("purchase_month")
    fig = px.area(rev, x="purchase_month", y="payment_value",
                  color_discrete_sequence=["#3498db"], template="plotly_white",
                  labels={"payment_value": "Revenue (R$)", "purchase_month": ""})
    fig.update_layout(margin=dict(l=20,r=20,t=30,b=40), xaxis_tickangle=-45, height=320)
    st.plotly_chart(fig, use_container_width=True)

with cr:
    st.subheader(" Status Order")
    sc = df_f["order_status"].value_counts().reset_index()
    sc.columns = ["Status", "Jumlah"]
    fig2 = px.pie(sc, values="Jumlah", names="Status", hole=0.4,
                  color_discrete_sequence=px.colors.qualitative.Set2, template="plotly_white")
    fig2.update_traces(textposition="inside", textinfo="percent+label")
    fig2.update_layout(showlegend=False, margin=dict(l=10,r=10,t=30,b=10), height=320)
    st.plotly_chart(fig2, use_container_width=True)

cl2, cr2 = st.columns([1.5, 1])
with cl2:
    st.subheader(" Top 15 Kategori Produk (by Revenue)")
    cat = (df_f.groupby("product_category_name")["payment_value"]
           .sum().sort_values(ascending=False).head(15).reset_index())
    cat.columns = ["Kategori", "Revenue"]
    fig3 = px.bar(cat.sort_values("Revenue"), x="Revenue", y="Kategori",
                  orientation="h", color="Revenue", color_continuous_scale="Blues",
                  template="plotly_white", labels={"Revenue": "Revenue (R$)"})
    fig3.update_layout(coloraxis_showscale=False, margin=dict(l=10,r=20,t=30,b=20), height=420)
    st.plotly_chart(fig3, use_container_width=True)

with cr2:
    st.subheader(" Metode Pembayaran")
    pt = df_f["payment_type"].value_counts().reset_index()
    pt.columns = ["Metode", "Jumlah"]
    fig4 = px.bar(pt, x="Jumlah", y="Metode", orientation="h", color="Metode",
                  color_discrete_sequence=px.colors.qualitative.Pastel, template="plotly_white")
    fig4.update_layout(showlegend=False, margin=dict(l=10,r=20,t=30,b=20), height=220)
    st.plotly_chart(fig4, use_container_width=True)

    st.subheader(" Review Score")
    rv = df_f["review_score"].dropna().value_counts().sort_index().reset_index()
    rv.columns = ["Score", "Jumlah"]
    cm = {1:"#e74c3c",2:"#e67e22",3:"#f1c40f",4:"#2ecc71",5:"#27ae60"}
    fig5 = px.bar(rv, x="Score", y="Jumlah", color="Score",
                  color_discrete_map=cm, template="plotly_white")
    fig5.update_layout(showlegend=False, margin=dict(l=10,r=10,t=30,b=20), height=190,
                        xaxis=dict(tickmode="linear",dtick=1))
    st.plotly_chart(fig5, use_container_width=True)

cl3, cr3 = st.columns(2)
with cl3:
    st.subheader(" Distribusi Waktu Pengiriman")
    dd = df_f["delivery_days"].dropna()
    dd = dd[(dd >= 0) & (dd <= 90)]
    fig6 = px.histogram(dd, nbins=45, color_discrete_sequence=["#9b59b6"],
                        template="plotly_white",
                        labels={"value":"Hari Pengiriman","count":"Jumlah Order"})
    fig6.update_layout(showlegend=False, margin=dict(l=10,r=10,t=30,b=20), height=310,
                        xaxis_title="Hari Pengiriman", yaxis_title="Jumlah Order")
    fig6.add_vline(x=dd.mean(), line_dash="dash", line_color="red",
                   annotation_text=f"  Rata-rata: {dd.mean():.1f} hari",
                   annotation_position="top right")
    st.plotly_chart(fig6, use_container_width=True)

with cr3:
    st.subheader(" Top 15 State by Order")
    so = (df_f.groupby("customer_state")["order_id"].nunique()
          .sort_values(ascending=False).head(15).reset_index())
    so.columns = ["State", "Orders"]
    fig7 = px.bar(so, x="State", y="Orders", color="Orders",
                  color_continuous_scale="Teal", template="plotly_white")
    fig7.update_layout(coloraxis_showscale=False, margin=dict(l=10,r=10,t=30,b=20), height=310)
    st.plotly_chart(fig7, use_container_width=True)

st.markdown("<hr>", unsafe_allow_html=True)
with st.expander(" Lihat Raw Data (10 baris pertama)"):
    st.dataframe(df_f.head(10), use_container_width=True)

st.markdown("""
<hr><p style=\'text-align:center; color:#bdc3c7; font-size:13px;\'>
Dashboard E-Commerce · Built with Streamlit & Plotly
</p>""", unsafe_allow_html=True)
'''

with open('dashboard.py', 'w') as f:
    f.write(dashboard_code)

print(' File dashboard.py berhasil dibuat')
!wc -l dashboard.py

 File dashboard.py berhasil dibuat
168 dashboard.py


## STEP 4 — Jalankan Streamlit + Cloudflared Tunnel



In [ ]:
import subprocess
import threading
import time
import re

# ── Jalankan Streamlit di background ────────────────────────────
def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'dashboard.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
        '--server.address', '0.0.0.0',
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()


print(' Menunggu Streamlit siap (5 detik)...')
time.sleep(5)
print('Streamlit berjalan di port 8501')


print('Membuka Cloudflared tunnel')
print('━' * 55)
print('Tunggu hingga muncul link trycloudflare.com')
print('━' * 55)


proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stderr=subprocess.PIPE,
    stdout=subprocess.DEVNULL,
    text=True
)


link_found = False
for line in proc.stderr:
    if 'trycloudflare.com' in line:
        # Ekstrak URL dari baris
        urls = re.findall(r'https://[\w\-]+\.trycloudflare\.com', line)
        if urls:
            print(f'Dashboard kamu LIVE di:')
            print(f'\n    {urls[0]} ')
            print(f'\n   Klik link di atas untuk membuka dashboard!')
            print('━' * 55)
            link_found = True
    if not link_found:
        # Tampilkan log lain kalau link belum muncul
        if 'INF' in line or 'error' in line.lower():
            print(line.strip())

proc.wait()

 Menunggu Streamlit siap (5 detik)...
Streamlit berjalan di port 8501
Membuka Cloudflared tunnel
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tunggu hingga muncul link trycloudflare.com
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-04-26T06:12:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-26T06:12:22Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-26T06:12:24Z INF +------------------------------------------